# Лабораторная 2. BeautifulSoup и Octoparse
Источник - Books to Scrape. В программной и no-code частях собраны по 40 книг с двух страниц.
По заданию в программной части нужно не менее 20 записей.

In [1]:
import csv
import html
from pathlib import Path
from IPython.display import HTML, display
from scraper import clean_text, parse_rating
from compare_results import main as compare_results

def show_table(rows, fields):
    head = "".join("<th>" + html.escape(field) + "</th>" for field in fields)
    body = "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(field, ""))) + "</td>"
                                    for field in fields) + "</tr>" for row in rows)
    display(HTML("<table><thead><tr>" + head + "</tr></thead><tbody>" + body + "</tbody></table>"))

with Path("books.csv").open(encoding="utf-8-sig", newline="") as file:
    books = list(csv.DictReader(file))
with Path("octoparse_books.csv").open(encoding="utf-8-sig", newline="") as file:
    octoparse = list(csv.DictReader(file))

## 1. Как выбираются поля
Карточка находится по `article.product_pod`. Название - `h3 a[title]`, цена - `p.price_color`,
рейтинг - класс у `p.star-rating`, следующая страница - `li.next a`.
Категория и UPC берутся со страницы книги. UPC используется как ключ удаления дублей.

In [2]:
show_table([
    {"Поле": "title", "Селектор": "h3 a", "Источник": "атрибут title"},
    {"Поле": "price_gbp", "Селектор": "p.price_color", "Источник": "текст цены"},
    {"Поле": "rating", "Селектор": "p.star-rating", "Источник": "класс One...Five"},
    {"Поле": "category", "Селектор": "ul.breadcrumb li", "Источник": "страница книги"},
    {"Поле": "upc", "Селектор": "table.table-striped", "Источник": "строка UPC"},
], ["Поле", "Селектор", "Источник"])

Поле,Селектор,Источник
title,h3 a,атрибут title
price_gbp,p.price_color,текст цены
rating,p.star-rating,класс One...Five
category,ul.breadcrumb li,страница книги
upc,table.table-striped,строка UPC


## 2. Очистка и типы
Повторяющиеся пробелы удаляются. Цена преобразуется в float, рейтинг - в целое число от 1 до 5.
CSV хранит текст, поэтому при чтении числовые поля нужно снова преобразовать.

In [3]:
print("Очистка:", clean_text("  A Light\n in the Attic  "))
print("Класс Three:", parse_rating(["star-rating", "Three"]))
print("Цена первой книги:", float(books[0]["price_gbp"]))
print("Тип цены:", type(float(books[0]["price_gbp"])).__name__)

Очистка: A Light in the Attic
Класс Three: 3
Цена первой книги: 51.77
Тип цены: float


## 3. Результат Python
Проверяются число строк, пустые ячейки и повторяющиеся UPC. Ниже показаны первые пять книг.

In [4]:
fields = list(books[0])
empty_cells = sum(not str(row[field]).strip() for row in books for field in fields)
duplicate_keys = len(books) - len({row["upc"] for row in books})
print("Строк:", len(books))
print("Дубликатов UPC:", duplicate_keys)
print("Доля пустых ячеек, %:", round(100 * empty_cells / (len(books) * len(fields)), 2))
show_table(books[:5], fields)

Строк: 40
Дубликатов UPC: 0
Доля пустых ячеек, %: 0.0


title,category,price_gbp,rating,url,upc
A Light in the Attic,Poetry,51.77,3,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,a897fe39b1053632
Tipping the Velvet,Historical Fiction,53.74,1,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,90fa61229261140a
Soumission,Fiction,50.1,1,https://books.toscrape.com/catalogue/soumission_998/index.html,6957f44c3847a760
Sharp Objects,Mystery,47.82,4,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,e00eb4fd7b871a48
Sapiens: A Brief History of Humankind,History,54.23,5,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,4165285e1663650f


## 4. Octoparse
Схема открывает каталог, обходит карточки и страницы книг, затем нажимает next.
Две итерации пагинации дали 40 записей. Скриншоты запуска и экспорта находятся в report.pdf.
В очищенном экспорте пять полей. Категория исключена, поскольку в исходном workflow она заполнена только для Poetry.

In [5]:
print("Octoparse: строк", len(octoparse), "полей", len(octoparse[0]))
show_table(octoparse[:5], list(octoparse[0]))

Octoparse: строк 40 полей 5


title,price_gbp,rating,url,upc
A Light in the Attic,51.77,3,https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,a897fe39b1053632
Tipping the Velvet,53.74,1,https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,90fa61229261140a
Soumission,50.1,1,https://books.toscrape.com/catalogue/soumission_998/index.html,6957f44c3847a760
Sharp Objects,47.82,4,https://books.toscrape.com/catalogue/sharp-objects_997/index.html,e00eb4fd7b871a48
Sapiens: A Brief History of Humankind,54.23,5,https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,4165285e1663650f


## 5. Сравнение
Таблицы сопоставляются по UPC. Сравниваются название, цена, рейтинг и URL.
Команда нового сбора: `python scraper.py --limit 40`. Она выполняется отдельно от этого просмотра сохранённых результатов.

In [6]:
compare_results()

BeautifulSoup: 40 строк, 6 полей, пропуски 0.00%
Поля: title, category, price_gbp, rating, url, upc
Octoparse: 40 строк, 5 полей, пропуски 0.00%
Поля: title, price_gbp, rating, url, upc
Общих UPC: 40
Только BeautifulSoup: 0
Только Octoparse: 0
Различий в title: 0
Различий в price_gbp: 0
Различий в rating: 0
Различий в url: 0
Различий в upc: 0


## Вывод
Получены одинаковые 40 книг без дублей и пропусков в итоговых таблицах.
В Python дополнительно сохранены категории всех книг.